In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [1]:
spark.sql("""
CREATE OR REPLACE TABLE gold_carrier_scorecard AS
SELECT
    carrier_name,
    carrier_id,
    MAX(sla_otdr_pct)                                                          AS sla_otdr_threshold,
    COUNT(*)                                                                    AS total_shipments,
    SUM(CASE WHEN shipment_status = 'Delivered' THEN 1 ELSE 0 END)             AS delivered_count,
    ROUND(AVG(CASE WHEN shipment_status='Delivered' THEN on_time_flag END)*100, 1) AS actual_otdr_pct,
    ROUND(SUM(freight_revenue_usd), 2)                                          AS total_revenue_usd,
    ROUND(SUM(freight_cost_usd), 2)                                             AS total_cost_usd,
    ROUND(SUM(gross_margin_usd), 2)                                             AS total_gross_margin_usd,
    ROUND(AVG(gross_margin_pct), 2)                                             AS avg_margin_pct,
    ROUND(AVG(cost_per_mile), 4)                                                AS avg_cost_per_mile,
    SUM(has_claim)                                                              AS total_claims,
    ROUND(SUM(claim_amount_usd), 2)                                             AS total_claim_amount_usd,
    ROUND(SUM(has_claim) * 100.0 / COUNT(*), 2)                                 AS claims_rate_pct,
    ROUND(AVG(transit_variance_days), 2)                                        AS avg_transit_variance_days
FROM silver_shipments
GROUP BY carrier_name, carrier_id
""")
print("gold_carrier_scorecard created")

StatementMeta(, c99259cd-ff83-445f-9848-5f393f3b208e, 3, Finished, Available, Finished, False)

gold_carrier_scorecard created


In [1]:
spark.sql("""
CREATE OR REPLACE TABLE gold_monthly_trend AS
SELECT
    pickup_year,
    pickup_month,
    CASE 
        WHEN pickup_month <= 3 THEN 'Q1'
        WHEN pickup_month <= 6 THEN 'Q2'
        WHEN pickup_month <= 9 THEN 'Q3'
        ELSE 'Q4'
    END AS pickup_quarter,
    DATE_FORMAT(pickup_date, 'yyyy-MM')     AS year_month,
    carrier_name,
    service_type,
    COUNT(*)                                AS total_shipments,
    ROUND(AVG(on_time_flag) * 100, 1)       AS otdr_pct,
    ROUND(SUM(freight_revenue_usd), 2)      AS total_revenue_usd,
    ROUND(SUM(gross_margin_usd), 2)         AS total_gross_margin_usd,
    ROUND(AVG(cost_per_mile), 4)            AS avg_cost_per_mile,
    SUM(has_claim)                          AS total_claims,
    ROUND(SUM(claim_amount_usd), 2)         AS total_claim_amount_usd
FROM silver_shipments
WHERE shipment_status = 'Delivered'
GROUP BY pickup_year, pickup_month, DATE_FORMAT(pickup_date, 'yyyy-MM'), carrier_name, service_type
""")
print("gold_monthly_trend created with pickup_quarter")

StatementMeta(, 91abcf96-e8c7-418e-9d20-6999462488be, 3, Finished, Available, Finished, False)

gold_monthly_trend created with pickup_quarter


In [3]:
spark.sql("""
CREATE OR REPLACE TABLE gold_lane_performance AS
SELECT
    lane,
    origin_city,
    destination_city,
    COUNT(*)                                    AS total_shipments,
    ROUND(AVG(distance_miles), 0)               AS avg_distance_miles,
    ROUND(AVG(on_time_flag) * 100, 1)           AS otdr_pct,
    ROUND(SUM(freight_revenue_usd), 2)          AS total_revenue_usd,
    ROUND(AVG(cost_per_mile), 4)                AS avg_cost_per_mile,
    ROUND(AVG(actual_transit_days), 1)          AS avg_actual_transit_days,
    ROUND(AVG(transit_variance_days), 2)        AS avg_transit_variance_days
FROM silver_shipments
WHERE shipment_status = 'Delivered'
GROUP BY lane, origin_city, destination_city
ORDER BY total_revenue_usd DESC
""")
print("gold_lane_performance created")

StatementMeta(, c99259cd-ff83-445f-9848-5f393f3b208e, 5, Finished, Available, Finished, False)

gold_lane_performance created


In [4]:
spark.sql("""
CREATE OR REPLACE TABLE gold_industry_analysis AS
SELECT
    industry,
    service_type,
    COUNT(*)                               AS shipments,
    ROUND(SUM(freight_revenue_usd), 2)     AS total_revenue_usd,
    ROUND(AVG(gross_margin_pct), 2)        AS avg_margin_pct,
    ROUND(AVG(on_time_flag) * 100, 1)      AS otdr_pct,
    SUM(has_claim)                         AS total_claims
FROM silver_shipments
GROUP BY industry, service_type
""")
print("gold_industry_analysis created")

StatementMeta(, c99259cd-ff83-445f-9848-5f393f3b208e, 6, Finished, Available, Finished, False)

gold_industry_analysis created
